# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their fields, using @id attributes

print("Available Record Sets:")
for recordset in dataset.record_sets:
    print(f"- RecordSet @id: {recordset.id}")
    print("  Fields:")
    for field in recordset.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, dataType: {field.data_type})")
    print("")
# Store record set and field @ids for further use
record_set_ids = [rs.id for rs in dataset.record_sets]
field_ids_dict = {rs.id: [f.id for f in rs.fields] for rs in dataset.record_sets}

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We'll use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using its @id
dataframes = {}
for record_set_id in record_set_ids:
    # Load all available records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded RecordSet: {record_set_id}")
        print(f"Fields (@id): {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"\nRecordSet {record_set_id} has no records.")
# For illustration, pick the first non-empty record set for further analysis
record_set_to_use = None
for rid in record_set_ids:
    if rid in dataframes:
        record_set_to_use = rid
        break
# If none loaded, inform user
if record_set_to_use:
    print(f"\nProceeding with RecordSet: {record_set_to_use}")
else:
    raise Exception('No record sets with data available for analysis.')

## 4. Exploratory Data Analysis (EDA)
We will process the selected record set: filter records, normalize a numeric field, and group by a categorical attribute, all using only `@id` column references.

In [ ]:
# Automatically detect a numeric field for demonstration
df = dataframes[record_set_to_use]
# Try to select the first float or integer field
numeric_field_id = None
for col in df.columns:
    # Check if the column is numeric (int or float) in dtype
    # Try to convert to numeric type to verify
    try:
        series = pd.to_numeric(df[col], errors='coerce')
        if series.notnull().sum() > 0 and (series.dtype == "int64" or series.dtype == "float64"):
            numeric_field_id = col
            break
    except Exception:
        continue

if not numeric_field_id:
    raise Exception(f"No numeric field found in RecordSet {record_set_to_use}.")
print(f"Using numeric field (by @id): {numeric_field_id}")

# Use an arbitrary threshold for filtering
threshold = df[numeric_field_id].astype(float).mean()
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another (categorical) field, e.g., the first non-numeric column
group_field_id = None
for col in df.columns:
    # Exclude numeric_field_id itself, and try for string/categorical
    if col != numeric_field_id and df[col].dtype == 'object':
        # Check for reasonably many unique values
        if df[col].nunique() > 1 and df[col].nunique() < len(df) / 2:
            group_field_id = col
            break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).agg({numeric_field_id: 'mean', f'{numeric_field_id}_normalized': 'mean'})
    print(f"\nGrouped data by {group_field_id} (by @id):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the grouping field, using only `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].astype(float), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id:
    # Boxplot grouped by the categorical field
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library, performing data loading, overview, basic analysis, and visualization. All dataset entities (record sets, fields, columns) were referenced by their `@id` attributes, ensuring consistent and reproducible workflows. This approach provides a robust basis for scalable, metadata-driven data science on complex FAIR datasets.